# PlantCLEF 2015 LeafScan S-CNN Training

Strict workflow for reproducing the paper protocol. Use the Google Drive LeafScan archive first, then smoke-train, train both S-CNN stages, and evaluate the full species ranking.


## 1. Runtime Check

Select `Runtime -> Change runtime type -> GPU` before running training cells.


In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Clone Or Update Project


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import yaml

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')

smoke_config = yaml.safe_load((PROJECT_DIR / 'configs/leafscan_smoke_training.yaml').read_text())
print('Smoke evaluation enabled:', smoke_config['evaluation']['enabled'])


## 3. Mount Google Drive, Unpack LeafScan Dataset, And Create Split

Expected archive paths: `/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz` and `/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz`. The strict paper run builds `leafscan_paper60_metadata.csv` from the 60 official test species; `leafscan_metadata_split.csv` is created only for smoke validation.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
%%bash
set -euo pipefail
trap 'echo "FAILED at line $LINENO: $BASH_COMMAND" >&2' ERR
export PYTHONUNBUFFERED=1
cd /content/diploma
ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz
echo "checking required archives"
ls -lh /content/drive/MyDrive/PlantCLEF2015*.tar.gz 2>/dev/null || true
if [ ! -f "$ARCHIVE" ]; then
  echo "Missing required LeafScan training archive: $ARCHIVE" >&2
  echo "Existing PlantCLEF archives in Google Drive:" >&2
  find /content/drive/MyDrive -maxdepth 1 -type f -iname 'PlantCLEF2015*.tar.gz' -printf '  %f\n' >&2 || true
  echo "Build it from the full PlantCLEF training package with:" >&2
  echo "  python scripts/build_plantclef_content_bundle.py --source-root /path/to/PlantCLEF2015/train --output $ARCHIVE --content LeafScan" >&2
  echo "Do not use PlantCLEF2015_leaf_only.tar.gz here; the paper protocol needs Content=LeafScan." >&2
  exit 2
fi
rm -rf data/plantclef2015
mkdir -p data/plantclef2015
echo "extracting training archive: $ARCHIVE"
tar -xzf "$ARCHIVE" -C data/plantclef2015
if [ ! -f data/plantclef2015/leafscan/metadata.csv ]; then
  echo "Training archive extracted, but leafscan/metadata.csv was not found." >&2
  echo "Archive top-level sample:" >&2
  tar -tzf "$ARCHIVE" | head -40 >&2
  exit 4
fi
cp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv
if [ ! -f "$TEST_ARCHIVE" ]; then
  echo "Missing required LeafScan test archive: $TEST_ARCHIVE" >&2
  echo "Run notebooks/plantclef_colab_test_data.ipynb first; strict paper training needs the official 60 test species list." >&2
  exit 3
fi
rm -rf data/plantclef2015/test_leafscan
mkdir -p data/plantclef2015/test_leafscan
echo "extracting test archive: $TEST_ARCHIVE"
tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
if [ ! -f data/plantclef2015/test_leafscan/leafscan/metadata.csv ]; then
  echo "Test archive extracted, but leafscan/metadata.csv was not found." >&2
  echo "Archive top-level sample:" >&2
  tar -tzf "$TEST_ARCHIVE" | head -40 >&2
  exit 5
fi
cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv
echo "test leafscan archive extracted"
plant-classifier-split-metadata \
  --metadata data/plantclef2015/leafscan_metadata.csv \
  --dataset-root data/plantclef2015/leafscan \
  --output data/plantclef2015/leafscan_metadata_split.csv \
  --train-ratio 0.70 \
  --val-ratio 0.15 \
  --test-ratio 0.15
wc -l data/plantclef2015/leafscan_metadata_split.csv
python - <<'PY2'
import csv
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image
with open('data/plantclef2015/leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    source_rows = list(csv.DictReader(file))
print('leafscan source rows:', len(source_rows))
print('leafscan source content:', Counter(row.get('content', '') for row in source_rows))
print('leafscan source genera:', len({row['genus'] for row in source_rows}))
print('leafscan source species:', len({row['species'] for row in source_rows}))
if len(source_rows) != 12605:
    raise RuntimeError(f'Expected 12605 PlantCLEF train LeafScan rows, got {len(source_rows)}')
with open('data/plantclef2015/leafscan_metadata_split.csv', newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
print('smoke split:', Counter(row['split'] for row in rows))

with open('data/plantclef2015/test_leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    test_rows = list(csv.DictReader(file))
test_species = {row['species'] for row in test_rows}
paper60_rows = [row for row in source_rows if row['species'] in test_species]
with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=source_rows[0].keys())
    writer.writeheader()
    writer.writerows(paper60_rows)
paper60_genera = {row['genus'] for row in paper60_rows}
paper60_species = {row['species'] for row in paper60_rows}
paper60_species_by_genus = defaultdict(set)
test_species_by_genus = defaultdict(set)
for row in paper60_rows:
    paper60_species_by_genus[row['genus']].add(row['species'])
for row in test_rows:
    test_species_by_genus[row['genus']].add(row['species'])
paper60_species_per_genus = {genus: len(species) for genus, species in paper60_species_by_genus.items()}
test_species_per_genus = {genus: len(species) for genus, species in test_species_by_genus.items()}
print('paper60 train rows:', len(paper60_rows))
print('paper60 train genera:', len(paper60_genera))
print('paper60 train species:', len(paper60_species))
print('paper60 max species per genus:', max(paper60_species_per_genus.values()))
print('paper60 genera with >6 species:', sorted(g for g, count in paper60_species_per_genus.items() if count > 6))
print('paper60 test rows:', len(test_rows))
print('paper60 test genera:', len({row['genus'] for row in test_rows}))
print('paper60 test species:', len(test_species))
print('paper60 test max species per genus:', max(test_species_per_genus.values()))
print('paper60 test genera with >6 species:', sorted(g for g, count in test_species_per_genus.items() if count > 6))
if len(paper60_rows) != 6527:
    print('warning: paper text/OCR suggests 6527 train rows, but this archive gives', len(paper60_rows))
if len(test_rows) != 221 or len({row['genus'] for row in test_rows}) != 43 or len(test_species) != 60:
    raise RuntimeError('Expected PlantCLEF paper test subset 221 rows / 43 genera / 60 species')
source_species = {row['species'] for row in source_rows}
missing_in_train = sorted(test_species - source_species)
if missing_in_train:
    print('paper60 missing test species in train:', missing_in_train[:20])
    raise RuntimeError(f'Cannot build paper60 train subset: {len(missing_in_train)} test species are missing from train metadata')
train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    print('paper60 species with fewer than 6 train images:', underfilled_species[:20])
    rows_by_species = defaultdict(list)
    for row in paper60_rows:
        rows_by_species[row['species']].append(row)
    augmented_dir = Path('data/plantclef2015/leafscan/augmented')
    augmented_dir.mkdir(parents=True, exist_ok=True)
    leafscan_root = Path('data/plantclef2015/leafscan')
    angles = [180, 90, 270, 15, -15]
    augmented_rows = []
    for species in underfilled_species:
        species_rows = rows_by_species[species]
        if not species_rows:
            raise RuntimeError(f'Cannot augment {species}: no train rows found')
        needed = 6 - len(species_rows)
        for index in range(needed):
            base_row = species_rows[index % len(species_rows)]
            source_path = Path(base_row['image_path'])
            if not source_path.is_absolute():
                source_path = leafscan_root / source_path
            angle = angles[index % len(angles)]
            output_name = f"{source_path.stem}_aug_rot{angle}_{index + 1}.jpg".replace('-', 'm')
            output_path = augmented_dir / output_name
            with Image.open(source_path) as image:
                image.convert('RGB').rotate(angle, expand=True, fillcolor=(255, 255, 255)).save(output_path, quality=95)
            augmented_row = dict(base_row)
            augmented_row['image_path'] = str(output_path.relative_to(leafscan_root))
            if 'source_xml' in augmented_row:
                augmented_row['source_xml'] = f"{augmented_row['source_xml']}#aug_rot{angle}"
            augmented_rows.append(augmented_row)
            print('augmented paper60 row:', species, 'from', source_path.name, '->', augmented_row['image_path'])
    paper60_rows.extend(augmented_rows)
    with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=source_rows[0].keys())
        writer.writeheader()
        writer.writerows(paper60_rows)
    print('paper60 augmented rows added:', len(augmented_rows))
train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    raise RuntimeError(f'Cannot build 6-shot paper subset after augmentation: {len(underfilled_species)} species have fewer than 6 train images')
print('paper60 six-shot training rows:', 6 * len(test_species))
PY2


## 4. Smoke Train `S-CNN (A)` Genus

This is only a pipeline check on a small subset. It also runs genus retrieval evaluation after the epoch.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_smoke_training.yaml   --stage genus   --output checkpoints/smoke_scnn_genus_vgg16.pt


## 5. Smoke Evaluate `S-CNN (A)` On Validation Split


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.eval_genus_cli   --config configs/leafscan_smoke_training.yaml   --checkpoint checkpoints/smoke_scnn_genus_vgg16.pt   --max-species 40   --references-per-genus 2   --queries-per-genus 2   --top-k 5


## 6. Full VGG16 Train `S-CNN (A)` Genus

Trains the global-view genus model with `configs/leafscan_paper60_training.yaml` and saves `scnn_genus_vgg16.pt` plus `scnn_genus_vgg16_best.pt`.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_paper60_training.yaml   --stage genus   --output checkpoints/scnn_genus_vgg16.pt


## 7. Save VGG16 Genus Checkpoints To Google Drive

Creates a new timestamped folder on every run, so previous checkpoint saves are preserved.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

MODEL=vgg16
STAGE=genus
BASE="/content/drive/MyDrive/diploma_checkpoints/leafscan_${MODEL}"
SAVE_ID="${STAGE}_$(date -u +%Y%m%dT%H%M%SZ)_${RANDOM}"
DEST="${BASE}/${SAVE_ID}"

test -d /content/drive/MyDrive || { echo "Google Drive is not mounted" >&2; exit 2; }
mkdir -p "$BASE"
mkdir "$DEST"

found=0
for file in "checkpoints/scnn_${STAGE}_${MODEL}.pt" "checkpoints/scnn_${STAGE}_${MODEL}_best.pt"; do
  if [ -f "$file" ]; then
    cp "$file" "$DEST/$(basename "$file")"
    found=1
  else
    echo "missing optional checkpoint: $file"
  fi
done

if [ "$found" -eq 0 ]; then
  echo "No ${STAGE} ${MODEL} checkpoints found to save" >&2
  exit 3
fi

date -u +%Y-%m-%dT%H:%M:%SZ > "$DEST/saved_at_utc.txt"
(git rev-parse --short HEAD || echo unknown) > "$DEST/commit.txt"
sha256sum "$DEST"/*.pt > "$DEST/sha256sums.txt"
echo "saved ${MODEL} ${STAGE} checkpoints to $DEST"
ls -lh "$DEST"


## 8. Evaluate Full VGG16 `S-CNN (A)`


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
REFERENCE_SEED=42
python -u -m plant_classifier.training.eval_genus_cli   --config configs/leafscan_paper60_training.yaml   --query-config configs/leafscan_test.yaml   --checkpoint checkpoints/scnn_genus_vgg16_best.pt   --max-species 0   --references-per-genus 6   --reference-level genus   --reference-seed "$REFERENCE_SEED"   --reference-split train   --score-mode comparator   --top-k 5 15 30 50


## 8a. Diagnostic VGG16 `S-CNN (A)` L1 Ranking

The paper-aligned score is the learned comparator above. This block keeps the same references and checkpoint, but ranks by raw L1 embedding distance to diagnose whether the comparator head is hurting retrieval.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
REFERENCE_SEED=42
python -u -m plant_classifier.training.eval_genus_cli   --config configs/leafscan_paper60_training.yaml   --query-config configs/leafscan_test.yaml   --checkpoint checkpoints/scnn_genus_vgg16_best.pt   --max-species 0   --references-per-genus 6   --reference-level genus   --reference-seed "$REFERENCE_SEED"   --reference-split train   --score-mode l1   --top-k 5 15 30 50


## 9. Full VGG16 Train `S-CNN (B)` Species

Run this after VGG16 `S-CNN (A)` has a reasonable genus retrieval result.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_paper60_training.yaml   --stage species   --output checkpoints/scnn_species_vgg16.pt


## 10. Save VGG16 Species Checkpoints To Google Drive

Creates a new timestamped folder on every run, so previous checkpoint saves are preserved.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

MODEL=vgg16
STAGE=species
BASE="/content/drive/MyDrive/diploma_checkpoints/leafscan_${MODEL}"
SAVE_ID="${STAGE}_$(date -u +%Y%m%dT%H%M%SZ)_${RANDOM}"
DEST="${BASE}/${SAVE_ID}"

test -d /content/drive/MyDrive || { echo "Google Drive is not mounted" >&2; exit 2; }
mkdir -p "$BASE"
mkdir "$DEST"

found=0
for file in "checkpoints/scnn_${STAGE}_${MODEL}.pt" "checkpoints/scnn_${STAGE}_${MODEL}_best.pt"; do
  if [ -f "$file" ]; then
    cp "$file" "$DEST/$(basename "$file")"
    found=1
  else
    echo "missing optional checkpoint: $file"
  fi
done

if [ "$found" -eq 0 ]; then
  echo "No ${STAGE} ${MODEL} checkpoints found to save" >&2
  exit 3
fi

date -u +%Y-%m-%dT%H:%M:%SZ > "$DEST/saved_at_utc.txt"
(git rev-parse --short HEAD || echo unknown) > "$DEST/commit.txt"
sha256sum "$DEST"/*.pt > "$DEST/sha256sums.txt"
echo "saved ${MODEL} ${STAGE} checkpoints to $DEST"
ls -lh "$DEST"


## 11. Evaluate Full VGG16 Two-Stage Species Ranking


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
REFERENCE_SEED=42
OUT_DIR="/content/drive/MyDrive/diploma_diagnostics/species_vgg16_$(date -u +%Y%m%dT%H%M%SZ)"
python -u -m plant_classifier.training.eval_species_cli \
  --config configs/leafscan_paper60_training.yaml \
  --query-config configs/leafscan_test.yaml \
  --genus-checkpoint checkpoints/scnn_genus_vgg16_best.pt \
  --species-checkpoint checkpoints/scnn_species_vgg16_best.pt \
  --genus-references-per-genus 6 \
  --references-per-species 6 \
  --genus-candidates 30 \
  --reference-seed "$REFERENCE_SEED" \
  --reference-split train \
  --genus-score-mode l1 \
  --species-score-mode comparator \
  --species-aggregation max \
  --output-dir "$OUT_DIR" \
  --top-k 1 3 5
ls -lh "$OUT_DIR"


## 12. Full VGG16 Baseline Species Classifier

Trains a plain supervised VGG16 classifier on the same six-shot paper60 LeafScan training subset as the S-CNN method. This is the comparison baseline without S-CNN pairs and without the two-stage hierarchy.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.classifier_cli \
  --config configs/leafscan_paper60_vgg16_baseline.yaml \
  --output checkpoints/vgg16_species_classifier.pt


## 13. Save VGG16 Baseline Classifier Checkpoints To Google Drive

Creates a new timestamped folder on every run, so previous classifier checkpoint saves are preserved.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

MODEL=vgg16
STAGE=species_classifier
BASE="/content/drive/MyDrive/diploma_checkpoints/leafscan_${MODEL}"
SAVE_ID="${STAGE}_$(date -u +%Y%m%dT%H%M%SZ)_${RANDOM}"
DEST="${BASE}/${SAVE_ID}"

test -d /content/drive/MyDrive || { echo "Google Drive is not mounted" >&2; exit 2; }
mkdir -p "$BASE"
mkdir "$DEST"

found=0
for file in "checkpoints/vgg16_species_classifier.pt" "checkpoints/vgg16_species_classifier_best.pt"; do
  if [ -f "$file" ]; then
    cp "$file" "$DEST/$(basename "$file")"
    found=1
  else
    echo "missing optional checkpoint: $file"
  fi
done

if [ "$found" -eq 0 ]; then
  echo "No ${STAGE} ${MODEL} checkpoints found to save" >&2
  exit 3
fi

date -u +%Y-%m-%dT%H:%M:%SZ > "$DEST/saved_at_utc.txt"
(git rev-parse --short HEAD || echo unknown) > "$DEST/commit.txt"
sha256sum "$DEST"/*.pt > "$DEST/sha256sums.txt"
echo "saved ${MODEL} ${STAGE} checkpoints to $DEST"
ls -lh "$DEST"


## 14. Evaluate VGG16 Baseline Species Classifier

Evaluates both best and final supervised VGG16 checkpoints on the official LeafScan test set.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

for checkpoint in \
  checkpoints/vgg16_species_classifier_best.pt \
  checkpoints/vgg16_species_classifier.pt; do
  echo "=== evaluating ${checkpoint} ==="
  name="$(basename "$checkpoint" .pt)"
  OUT_DIR="/content/drive/MyDrive/diploma_diagnostics/classifier_${name}_$(date -u +%Y%m%dT%H%M%SZ)"
  python -u -m plant_classifier.training.eval_classifier_cli \
    --config configs/leafscan_paper60_vgg16_baseline.yaml \
    --query-config configs/leafscan_test.yaml \
    --checkpoint "$checkpoint" \
    --output-dir "$OUT_DIR" \
    --top-k 1 3 5
  ls -lh "$OUT_DIR"
done


## 15. Smoke Train EfficientNet-B3

Checks both genus and species stages on a small subset before the full EfficientNet-B3 run.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/smoke_training_efficientnet_b3.yaml   --stage genus   --output checkpoints/smoke_scnn_genus_efficientnet_b3.pt
python -u -m plant_classifier.training.cli   --config configs/smoke_training_efficientnet_b3.yaml   --stage species   --output checkpoints/smoke_scnn_species_efficientnet_b3.pt


## 16. Smoke Evaluate EfficientNet-B3 `S-CNN (A)` On Validation Split


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.eval_genus_cli   --config configs/smoke_training_efficientnet_b3.yaml   --checkpoint checkpoints/smoke_scnn_genus_efficientnet_b3.pt   --max-species 40   --references-per-genus 2   --queries-per-genus 2   --top-k 5


## 17. Full EfficientNet-B3 Train `S-CNN (A)` Genus


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_paper60_efficientnet_b3.yaml   --stage genus   --output checkpoints/scnn_genus_efficientnet_b3.pt


## 18. Save EfficientNet-B3 Genus Checkpoints To Google Drive

Creates a new timestamped folder on every run, so previous checkpoint saves are preserved.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

MODEL=efficientnet_b3
STAGE=genus
BASE="/content/drive/MyDrive/diploma_checkpoints/leafscan_${MODEL}"
SAVE_ID="${STAGE}_$(date -u +%Y%m%dT%H%M%SZ)_${RANDOM}"
DEST="${BASE}/${SAVE_ID}"

test -d /content/drive/MyDrive || { echo "Google Drive is not mounted" >&2; exit 2; }
mkdir -p "$BASE"
mkdir "$DEST"

found=0
for file in "checkpoints/scnn_${STAGE}_${MODEL}.pt" "checkpoints/scnn_${STAGE}_${MODEL}_best.pt"; do
  if [ -f "$file" ]; then
    cp "$file" "$DEST/$(basename "$file")"
    found=1
  else
    echo "missing optional checkpoint: $file"
  fi
done

if [ "$found" -eq 0 ]; then
  echo "No ${STAGE} ${MODEL} checkpoints found to save" >&2
  exit 3
fi

date -u +%Y-%m-%dT%H:%M:%SZ > "$DEST/saved_at_utc.txt"
(git rev-parse --short HEAD || echo unknown) > "$DEST/commit.txt"
sha256sum "$DEST"/*.pt > "$DEST/sha256sums.txt"
echo "saved ${MODEL} ${STAGE} checkpoints to $DEST"
ls -lh "$DEST"


## 19. Evaluate Full EfficientNet-B3 `S-CNN (A)`


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
REFERENCE_SEED=42
python -u -m plant_classifier.training.eval_genus_cli \
  --config configs/leafscan_paper60_efficientnet_b3.yaml \
  --query-config configs/leafscan_test.yaml \
  --checkpoint checkpoints/scnn_genus_efficientnet_b3_best.pt \
  --max-species 0 \
  --references-per-genus 6 \
  --reference-level genus \
  --reference-seed "$REFERENCE_SEED" \
  --reference-split train \
  --score-mode comparator \
  --top-k 5 15 30 50


## 20. Full EfficientNet-B3 Train `S-CNN (B)` Species


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_paper60_efficientnet_b3.yaml   --stage species   --output checkpoints/scnn_species_efficientnet_b3.pt


## 21. Save EfficientNet-B3 Species Checkpoints To Google Drive

Creates a new timestamped folder on every run, so previous checkpoint saves are preserved.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

MODEL=efficientnet_b3
STAGE=species
BASE="/content/drive/MyDrive/diploma_checkpoints/leafscan_${MODEL}"
SAVE_ID="${STAGE}_$(date -u +%Y%m%dT%H%M%SZ)_${RANDOM}"
DEST="${BASE}/${SAVE_ID}"

test -d /content/drive/MyDrive || { echo "Google Drive is not mounted" >&2; exit 2; }
mkdir -p "$BASE"
mkdir "$DEST"

found=0
for file in "checkpoints/scnn_${STAGE}_${MODEL}.pt" "checkpoints/scnn_${STAGE}_${MODEL}_best.pt"; do
  if [ -f "$file" ]; then
    cp "$file" "$DEST/$(basename "$file")"
    found=1
  else
    echo "missing optional checkpoint: $file"
  fi
done

if [ "$found" -eq 0 ]; then
  echo "No ${STAGE} ${MODEL} checkpoints found to save" >&2
  exit 3
fi

date -u +%Y-%m-%dT%H:%M:%SZ > "$DEST/saved_at_utc.txt"
(git rev-parse --short HEAD || echo unknown) > "$DEST/commit.txt"
sha256sum "$DEST"/*.pt > "$DEST/sha256sums.txt"
echo "saved ${MODEL} ${STAGE} checkpoints to $DEST"
ls -lh "$DEST"


## 22. Evaluate Full EfficientNet-B3 Two-Stage Species Ranking


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
REFERENCE_SEED=42
OUT_DIR="/content/drive/MyDrive/diploma_diagnostics/species_efficientnet_b3_$(date -u +%Y%m%dT%H%M%SZ)"
python -u -m plant_classifier.training.eval_species_cli \
  --config configs/leafscan_paper60_efficientnet_b3.yaml \
  --query-config configs/leafscan_test.yaml \
  --genus-checkpoint checkpoints/scnn_genus_efficientnet_b3_best.pt \
  --species-checkpoint checkpoints/scnn_species_efficientnet_b3_best.pt \
  --genus-references-per-genus 6 \
  --references-per-species 6 \
  --genus-candidates 30 \
  --reference-seed "$REFERENCE_SEED" \
  --reference-split train \
  --genus-score-mode l1 \
  --species-score-mode comparator \
  --species-aggregation max \
  --output-dir "$OUT_DIR" \
  --top-k 1 3 5
ls -lh "$OUT_DIR"



## 23. Smoke Train MobileNetV3-Large

Checks both genus and species stages on a small subset before the full MobileNetV3-Large run.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/smoke_training_mobilenet_v3_large.yaml   --stage genus   --output checkpoints/smoke_scnn_genus_mobilenet_v3_large.pt
python -u -m plant_classifier.training.cli   --config configs/smoke_training_mobilenet_v3_large.yaml   --stage species   --output checkpoints/smoke_scnn_species_mobilenet_v3_large.pt


## 24. Smoke Evaluate MobileNetV3-Large `S-CNN (A)` On Validation Split


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.eval_genus_cli   --config configs/smoke_training_mobilenet_v3_large.yaml   --checkpoint checkpoints/smoke_scnn_genus_mobilenet_v3_large.pt   --max-species 40   --references-per-genus 2   --queries-per-genus 2   --top-k 5


## 25. Full MobileNetV3-Large Train `S-CNN (A)` Genus


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_paper60_mobilenet_v3_large.yaml   --stage genus   --output checkpoints/scnn_genus_mobilenet_v3_large.pt


## 26. Save MobileNetV3-Large Genus Checkpoints To Google Drive

Creates a new timestamped folder on every run, so previous checkpoint saves are preserved.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

MODEL=mobilenet_v3_large
STAGE=genus
BASE="/content/drive/MyDrive/diploma_checkpoints/leafscan_${MODEL}"
SAVE_ID="${STAGE}_$(date -u +%Y%m%dT%H%M%SZ)_${RANDOM}"
DEST="${BASE}/${SAVE_ID}"

test -d /content/drive/MyDrive || { echo "Google Drive is not mounted" >&2; exit 2; }
mkdir -p "$BASE"
mkdir "$DEST"

found=0
for file in "checkpoints/scnn_${STAGE}_${MODEL}.pt" "checkpoints/scnn_${STAGE}_${MODEL}_best.pt"; do
  if [ -f "$file" ]; then
    cp "$file" "$DEST/$(basename "$file")"
    found=1
  else
    echo "missing optional checkpoint: $file"
  fi
done

if [ "$found" -eq 0 ]; then
  echo "No ${STAGE} ${MODEL} checkpoints found to save" >&2
  exit 3
fi

date -u +%Y-%m-%dT%H:%M:%SZ > "$DEST/saved_at_utc.txt"
(git rev-parse --short HEAD || echo unknown) > "$DEST/commit.txt"
sha256sum "$DEST"/*.pt > "$DEST/sha256sums.txt"
echo "saved ${MODEL} ${STAGE} checkpoints to $DEST"
ls -lh "$DEST"


## 27. Evaluate Full MobileNetV3-Large `S-CNN (A)`


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
REFERENCE_SEED=42
python -u -m plant_classifier.training.eval_genus_cli \
  --config configs/leafscan_paper60_mobilenet_v3_large.yaml \
  --query-config configs/leafscan_test.yaml \
  --checkpoint checkpoints/scnn_genus_mobilenet_v3_large_best.pt \
  --max-species 0 \
  --references-per-genus 6 \
  --reference-level genus \
  --reference-seed "$REFERENCE_SEED" \
  --reference-split train \
  --score-mode comparator \
  --top-k 5 15 30 50


## 28. Full MobileNetV3-Large Train `S-CNN (B)` Species


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_paper60_mobilenet_v3_large.yaml   --stage species   --output checkpoints/scnn_species_mobilenet_v3_large.pt


## 29. Save MobileNetV3-Large Species Checkpoints To Google Drive

Creates a new timestamped folder on every run, so previous checkpoint saves are preserved.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

MODEL=mobilenet_v3_large
STAGE=species
BASE="/content/drive/MyDrive/diploma_checkpoints/leafscan_${MODEL}"
SAVE_ID="${STAGE}_$(date -u +%Y%m%dT%H%M%SZ)_${RANDOM}"
DEST="${BASE}/${SAVE_ID}"

test -d /content/drive/MyDrive || { echo "Google Drive is not mounted" >&2; exit 2; }
mkdir -p "$BASE"
mkdir "$DEST"

found=0
for file in "checkpoints/scnn_${STAGE}_${MODEL}.pt" "checkpoints/scnn_${STAGE}_${MODEL}_best.pt"; do
  if [ -f "$file" ]; then
    cp "$file" "$DEST/$(basename "$file")"
    found=1
  else
    echo "missing optional checkpoint: $file"
  fi
done

if [ "$found" -eq 0 ]; then
  echo "No ${STAGE} ${MODEL} checkpoints found to save" >&2
  exit 3
fi

date -u +%Y-%m-%dT%H:%M:%SZ > "$DEST/saved_at_utc.txt"
(git rev-parse --short HEAD || echo unknown) > "$DEST/commit.txt"
sha256sum "$DEST"/*.pt > "$DEST/sha256sums.txt"
echo "saved ${MODEL} ${STAGE} checkpoints to $DEST"
ls -lh "$DEST"


## 30. Evaluate Full MobileNetV3-Large Two-Stage Species Ranking


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
REFERENCE_SEED=42
OUT_DIR="/content/drive/MyDrive/diploma_diagnostics/species_mobilenet_v3_large_$(date -u +%Y%m%dT%H%M%SZ)"
python -u -m plant_classifier.training.eval_species_cli \
  --config configs/leafscan_paper60_mobilenet_v3_large.yaml \
  --query-config configs/leafscan_test.yaml \
  --genus-checkpoint checkpoints/scnn_genus_mobilenet_v3_large_best.pt \
  --species-checkpoint checkpoints/scnn_species_mobilenet_v3_large_best.pt \
  --genus-references-per-genus 6 \
  --references-per-species 6 \
  --genus-candidates 30 \
  --reference-seed "$REFERENCE_SEED" \
  --reference-split train \
  --genus-score-mode l1 \
  --species-score-mode comparator \
  --species-aggregation max \
  --output-dir "$OUT_DIR" \
  --top-k 1 3 5
ls -lh "$OUT_DIR"



## 31. Sync Checkpoints To Google Drive

Old Drive checkpoints are removed unless their name contains `_best`. Local `*_best` files are copied to Drive as regular checkpoint names, so Drive `*_best` files can mean best-across-all-runs.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python scripts/sync_checkpoints_to_drive.py \
  --source checkpoints \
  --dest /content/drive/MyDrive/diploma_checkpoints \
  --keep-token _best
ls -lh /content/drive/MyDrive/diploma_checkpoints


## 32. Build VGG16 Reference Index For Desktop Inference


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.build_index_cli \
  --config configs/leafscan_paper60_training.yaml \
  --genus-checkpoint checkpoints/scnn_genus_vgg16_best.pt \
  --species-checkpoint checkpoints/scnn_species_vgg16_best.pt \
  --output checkpoints/reference_index_leafscan_vgg16.pt
ls -lh checkpoints


## 33. Final Sync To Google Drive


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python scripts/sync_checkpoints_to_drive.py \
  --source checkpoints \
  --dest /content/drive/MyDrive/diploma_checkpoints \
  --keep-token _best
ls -lh /content/drive/MyDrive/diploma_checkpoints
